# DeepSeek API 流式输出解析 - OpenAI SDK

> 指导文章：[DeepSeek API 流式输出解析 - OpenAI SDK](https://github.com/Hoper-J/AI-Guide-and-Demos-zh_CN/blob/master/Guide/DeepSeek%20API%20流式输出解析%20-%20OpenAI%20SDK.md)

从下方选择平台开始，替换 `your-api-key` 后点击 `►` 或使用 `Shift + 回车` 运行代码块。

**注意**：

- `completion` 遍历一次后就会被消耗掉，如果需要对同一份流式输出数据进行多次操作，可以在循环中同时处理。

  为了正确演示，代码文件中的每个示例都会启动一次新的对话。

在线链接：[Kaggle](https://www.kaggle.com/code/aidemos/deepseek-api-guide-3) | [Colab](https://colab.research.google.com/drive/1qZ1Ihc2XjD91tocjjrCp5cHVq4_WDBVd?usp=sharing)

# 环境依赖

In [1]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


# DeepSeek

## 认识流式输出

### 非思考模式

In [2]:
from openai import OpenAI
import os

client = OpenAI(
    api_key="your-api-key",
    base_url="https://api.deepseek.com",
)

# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
from pprint import pprint
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916248,
 'id': 'f9ee4597-6ead-4d32-85a5-612d99ad7948',
 'model': 'deepseek-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
 'usage': None}


{'choices': [{'delta': {'content': '我是',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916248,
 'id': 'f9ee4597-6ead-4d32-85a5-612d99ad7948',
 'model': 'deepseek-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
 'usage': None}
{'choices': [{'delta': {'content': 'Deep',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916248,
 'id': 'f9ee4597-6ead-4d32-85a5-612d99ad7948',
 'model': 'deepseek-flash',
 'object': 'chat.completion.chunk',
 'service_ti

#### 实时打印回复

In [3]:
completion = client.chat.completions.create(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    # 判断回复内容是否非空
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end='')  # 设置 end='' 实现不换行，视觉上拼接输出

我是一个由深度求索公司创造的AI助手，随时为你解答

问题、提供帮助。

#### 拼接回复

In [4]:
completion = client.chat.completions.create(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
content = ""
for chunk in completion:
    if chunk.choices[0].delta.content:
    	content += chunk.choices[0].delta.content
print(content)

我是DeepSeek，由深度求索公司创造的AI助手，致力于为你提供热情、细腻的帮助！


### 思考模式

同一数据块可能同时包含推理末尾和正文开头，下面分别读取两个字段，使用独立的 `if` 打印或拼接，避免漏字。

In [5]:
# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': None,
                        'function_call': None,
                        'reasoning_content': '',
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916250,
 'id': '8a25b09b-741b-47af-ae47-6ec9805aa1a5',
 'model': 'deepseek-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
 'usage': None}


{'choices': [{'delta': {'content': None,
                        'function_call': None,
                        'reasoning_content': 'The',
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916250,
 'id': '8a25b09b-741b-47af-ae47-6ec9805aa1a5',
 'model': 'deepseek-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
 'usage': None}
{'choices': [{'delta': {'content': None,
                        'function_call': None,
                        'reasoning_content': ' user',
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916250,
 'id': '8a25b09b-74

{'choices': [{'delta': {'content': None,
                        'function_call': None,
                        'reasoning_content': '.',
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916250,
 'id': '8a25b09b-741b-47af-ae47-6ec9805aa1a5',
 'model': 'deepseek-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
 'usage': None}
{'choices': [{'delta': {'content': '我是',
                        'function_call': None,
                        'reasoning_content': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916250,
 'id': '8a25b09b-741b-47

#### 实时打印回复

In [6]:
completion = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True,  # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        print(reasoning, end='')
    if delta.content:
        print(delta.content, end='')


The user is asking me to introduce myself in one sentence, in Chinese. I should respond in Chinese with a single sentence.我是DeepSeek，一个由深度求索公司打造的AI助手，乐于为你解答问题、提供帮助。

#### 拼接回复

In [7]:
completion = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
reasoning_content = ""
content = ""
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        reasoning_content += reasoning
    if delta.content:
        content += delta.content
print(f"===== 模型推理过程 =====\n{reasoning_content}")
print(f"===== 模型回复 =====\n{content}")

===== 模型推理过程 =====
我们需要回答中文一句话介绍自己。需要简洁。用户问“请用一句话介绍你自己”。我们作为助手，一句话即可。要符合身份。例如：“我是由深度求索公司开发的AI助手，乐于为你提供信息与帮助。” 注意不要声称？ 我是 DeepSeek? 系统没给具体身份，但通常可以答AI助手。一句话。保持自然。用中文。
===== 模型回复 =====
我是一个乐于助人的AI助手，愿意为你解答问题、提供信息与帮助。


# 硅基流动

## 认识流式输出

### 非思考模式

In [8]:
from openai import OpenAI
import os

client = OpenAI(
    api_key="your-api-key", # 1：替换成对应的 API_Key，可以使用环境变量而非明文填写，即 api_key=os.getenv("SILICONFLOW_API_KEY")
    base_url="https://api.siliconflow.cn/v1", # 2：每个平台的 base_url 不同
)

# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V4.1-Flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
from pprint import pprint
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916244,
 'id': '01a17222c6ae068f39f1fba42d4746cf',
 'model': 'deepseek-ai/DeepSeek-V4.1-Flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': '',
 'usage': {'completion_tokens': 0,
           'completion_tokens_details': {'accepted_prediction_tokens': None,
                                         'audio_tokens': None,
                                         'reasoning_tokens': 0,
                                         'rejected_prediction_tokens': None},
           'prompt_cache_hit_tokens': 0,
           'prompt_cache_miss_tokens': 16,
           'prompt_tokens': 16,
           'prompt_tokens_details': {'audio_tokens': No

{'choices': [{'delta': {'content': '。',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916244,
 'id': '01a17222c6ae068f39f1fba42d4746cf',
 'model': 'deepseek-ai/DeepSeek-V4.1-Flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': '',
 'usage': {'completion_tokens': 19,
           'completion_tokens_details': {'accepted_prediction_tokens': None,
                                         'audio_tokens': None,
                                         'reasoning_tokens': 0,
                                         'rejected_prediction_tokens': None},
           'prompt_cache_hit_tokens': 0,
           'prompt_cache_miss_tokens': 16,
           'prompt_tokens': 16,
           'prompt_tokens_details': {'audio_tokens': None, '

#### 实时打印回复

In [9]:
completion = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V4.1-Flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    # 判断回复内容是否非空
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end='')  # 设置 end='' 实现不换行，视觉上拼接输出

我是一个由深度求索公司创造的AI助手，随时

准备为你解答问题并提供帮助。

#### 拼接回复

In [10]:
completion = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V4.1-Flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
content = ""
for chunk in completion:
    if chunk.choices[0].delta.content:
    	content += chunk.choices[0].delta.content
print(content)

我是一个由DeepSeek最新版模型驱动的AI助手，乐于为你解答问题、提供信息和帮助。


### 思考模式

In [11]:
# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V4.1-Flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "enabled"}},  # 硅基流动上偶尔不默认思考，所以这里显式开启
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916331,
 'id': '01a1722419c9656826af560f7b990b16',
 'model': 'deepseek-ai/DeepSeek-V4.1-Flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': '',
 'usage': {'completion_tokens': 0,
           'completion_tokens_details': {'accepted_prediction_tokens': None,
                                         'audio_tokens': None,
                                         'reasoning_tokens': 0,
                                         'rejected_prediction_tokens': None},
           'prompt_cache_hit_tokens': 0,
           'prompt_cache_miss_tokens': 41,
           'prompt_tokens': 41,
           'prompt_tokens_details': {'audio_tokens': No

#### 实时打印回复

In [12]:
completion = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V4.1-Flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "enabled"}},  # 硅基流动上偶尔不默认思考，所以这里显式开启
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        print(reasoning, end='')
    if delta.content:
        print(delta.content, end='')


The user asks: "请用一句话介绍

你自己" — introduce yourself in one sentence. Keep it concise, one sentence, in Chinese.我是一名由深度求索公司开发的 AI 助手，乐于用清晰、有用的方式帮你解答问题、处理文字

与思路整理。

#### 拼接回复

In [13]:
completion = client.chat.completions.create(
    model="deepseek-ai/DeepSeek-V4.1-Flash",
    extra_body={"thinking": {"type": "enabled"}},  # 硅基流动上偶尔不默认思考，所以这里显式开启
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
reasoning_content = ""
content = ""
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        reasoning_content += reasoning
    if delta.content:
        content += delta.content
print(f"===== 模型推理过程 =====\n{reasoning_content}")
print(f"===== 模型回复 =====\n{content}")

===== 模型推理过程 =====
我们需要回答中文，一句话介绍自己。需要简洁。作为助手。一句话。可以：我是由深度求索公司创造的AI助手，乐于为你提供信息、解答问题和协助完成任务。注意不要过度。只需一句话。用户中文。回答一句话。确保介绍自己。可提及DeepSeek? 系统说AI assistant accessed via API. 用一句话介绍自己。中文。最终只一句。
===== 模型回复 =====
我是由深度求索公司创造的AI助手，乐于为你解答问题、提供信息并协助完成各类任务。


# 阿里云百炼

## 认识流式输出

### 非思考模式

In [14]:
from openai import OpenAI
import os

client = OpenAI(
    api_key="your-api-key", # 1：替换成对应的 API_Key，可以使用环境变量而非明文填写，即 api_key=os.getenv("SILICONFLOW_API_KEY")
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", # 2：每个平台的 base_url 不同
)

# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
from pprint import pprint
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916261,
 'id': 'chatcmpl-da0d8252-c17d-907c-ab5b-a321e95f5197',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': '我是',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916261,
 'id': 'chatcmpl-da0d8252-c17d-907c-ab5b-a321e95f5197',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_t

{'choices': [{'delta': {'content': '深度求索公司创造的AI',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916261,
 'id': 'chatcmpl-da0d8252-c17d-907c-ab5b-a321e95f5197',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': '助手，致力于',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916261,
 'id': 'chatcmpl-da0d8252-c17d-907c-ab5b-a321e95f5197',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 's

{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': 'stop',
              'index': 0,
              'logprobs': None}],
 'created': 1792916261,
 'id': 'chatcmpl-da0d8252-c17d-907c-ab5b-a321e95f5197',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}


#### 实时打印回复

In [15]:
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    # 判断回复内容是否非空
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end='')  # 设置 end='' 实现不换行，视觉上拼接输出

我是DeepSeek，一个由深度求索公司创造的AI助手，随时准备为你提供帮助和

解答问题！

#### 拼接回复

In [16]:
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
content = ""
for chunk in completion:
    if chunk.choices[0].delta.content:
    	content += chunk.choices[0].delta.content
print(content)

我是DeepSeek，由深度求索公司创造的AI助手，致力于为你提供热情、细腻的帮助！


### 思考模式

In [17]:
# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'reasoning_content': '',
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916264,
 'id': 'chatcmpl-13eddcf4-8dbb-9bb6-ba8d-7a7f48de4d78',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': None,
                        'function_call': None,
                        'reasoning_content': 'The',
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916264,
 'id': 'chatcmpl-13eddcf4-8dbb-9bb6

{'choices': [{'delta': {'content': None,
                        'function_call': None,
                        'reasoning_content': ' respond in one sentence,',
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916264,
 'id': 'chatcmpl-13eddcf4-8dbb-9bb6-ba8d-7a7f48de4d78',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': None,
                        'function_call': None,
                        'reasoning_content': ' in Chinese.',
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916264,
 'id':

{'choices': [{'delta': {'content': '深度求索公司研发',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916264,
 'id': 'chatcmpl-13eddcf4-8dbb-9bb6-ba8d-7a7f48de4d78',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': '的 AI 助手，随时',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916264,
 'id': 'chatcmpl-13eddcf4-8dbb-9bb6-ba8d-7a7f48de4d78',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 '

#### 实时打印回复

In [18]:
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        print(reasoning, end='')
    if delta.content:
        print(delta.content, end='')


我们需要回答用户中文请求：“请用一句话介绍你自己

”。需要一句话。应该用中文，简洁。可以：我是由

深度求索公司开发的AI助手，乐于用中文为你提供清晰、可靠的帮助。需要一句话。注意不要提及具体版本？可以。确保一句话。最终只输出一句话。我是由深度求索公司开发的AI助手，乐于用中文为你提供清晰、可靠的帮助。

#### 拼接回复

In [19]:
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash",
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
reasoning_content = ""
content = ""
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        reasoning_content += reasoning
    if delta.content:
        content += delta.content
print(f"===== 模型推理过程 =====\n{reasoning_content}")
print(f"===== 模型回复 =====\n{content}")

===== 模型推理过程 =====
The user asks in Chinese: "Please introduce yourself in one sentence." I should respond in one sentence, in Chinese.
===== 模型回复 =====
我是由深度求索公司创造的DeepSeek，一个乐于为你解答问题、提供帮助的AI助手。


# 百度智能云

## 认识流式输出

### 非思考模式

In [20]:
from openai import OpenAI
import os

client = OpenAI(
    api_key="your-api-key", # 1：替换成对应的 API_Key，可以使用环境变量而非明文填写，即 api_key=os.getenv("SILICONFLOW_API_KEY")
    base_url="https://qianfan.baidubce.com/v2" # 2：每个平台的 base_url 不同
)

# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
from pprint import pprint
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': '我是',
                        'function_call': None,
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'flag': 0,
              'index': 0,
              'logprobs': None}],
 'created': 1792916268,
 'id': 'as-4jx71au9p6',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': 'Deep',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'flag': 0,
              'index': 0,
              'logprobs': None}],
 'created': 1792916268,
 'id': 'as-4jx71au9p6',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None

{'choices': [{'delta': {'content': '、',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'flag': 0,
              'index': 0,
              'logprobs': None}],
 'created': 1792916268,
 'id': 'as-4jx71au9p6',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': '细腻的方式帮你',
                        'function_call': None,
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'flag': 0,
              'index': 0,
              'logprobs': None}],
 'created': 1792916268,
 'id': 'as-4jx71au9p6',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 's

#### 实时打印回复

In [21]:
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    # 判断回复内容是否非空
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end='')  # 设置 end='' 实现不换行，视觉上拼接输出

我是你的智能助手，可以回答问题、提供建议，并尽力帮助你解决各种问题。

#### 拼接回复

In [22]:
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
content = ""
for chunk in completion:
    if chunk.choices[0].delta.content:
    	content += chunk.choices[0].delta.content
print(content)

我是DeepSeek，一个由深度求索公司打造的AI助手，随时准备为你提供帮助、解答疑问！


### 思考模式

In [23]:
# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': None,
                        'function_call': None,
                        'reasoning_content': 'The',
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'flag': 0,
              'index': 0,
              'logprobs': None}],
 'created': 1792916341,
 'id': 'as-u8crk7rjbi',
 'model': 'deepseek-v4.1-flash',
 'object': 'chat.completion.chunk',
 'service_tier': None,
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': None,
                        'function_call': None,
                        'reasoning_content': ' user is',
                        'refusal': None,
                        'role': None,
                        'tool_calls': None},
              'finish_reason': None,
              'flag': 0,
              'index': 0,
              'logprobs': None}],
 'created': 1792916341,
 'id': 

#### 实时打印回复

In [24]:
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash", # 3：模型标识（model_id）可能存在差异
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        print(reasoning, end='')
    if delta.content:
        print(delta.content, end='')


我们需要回答用户中文请求：“请用一句话介绍你自己”。需要一句话。可以简洁。需要作为AI助手介绍。注意“一句话”所以只输出一句话，不要多句。可以：“我是一个乐于助人、随时为你提供信息和解答问题的AI助手。” 这是中文一句话。可以包含我是谁。需要符合。最终只一句。我是一个乐于助人、随时为你提供信息和解答问题的AI助手。

#### 拼接回复

In [25]:
completion = client.chat.completions.create(
    model="deepseek-v4.1-flash",
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
reasoning_content = ""
content = ""
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        reasoning_content += reasoning
    if delta.content:
        content += delta.content
print(f"===== 模型推理过程 =====\n{reasoning_content}")
print(f"===== 模型回复 =====\n{content}")

===== 模型推理过程 =====
We need answer in Chinese, one sentence. Need introduce self. Simple. Ensure one sentence. "我是一个乐于助人、知识广泛且注重准确与安全的AI助手，随时准备用中文为你解答问题、提供思路和帮助。" That's one sentence.
===== 模型回复 =====
我是一个乐于助人、知识广泛且注重准确与安全的AI助手，随时准备用中文为你解答问题、提供思路和帮助。


# 字节火山引擎

## 认识流式输出

### 非思考模式

In [26]:
from openai import OpenAI
import os

client = OpenAI(
    api_key="your-api-key", # 1：替换成对应的 API_Key，可以使用环境变量而非明文填写，即 api_key=os.getenv("SILICONFLOW_API_KEY")
    base_url="https://ark.cn-beijing.volces.com/api/v3" # 2：每个平台的 base_url 不同
)

# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-v4-1-flash-261017", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
from pprint import pprint
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916277,
 'id': '021792916276720c65202e590747a44f6b901a7b0655cff1a8bf6',
 'model': 'deepseek-v4-1-flash-261017',
 'object': 'chat.completion.chunk',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': '我是',
                        'function_call': None,
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916277,
 'id': '021792916276720c65202e590747a44f6b901a7b0655cff1a8bf6',
 'model': 'deepseek-v4-1-flash-261017',
 'obj

{'choices': [{'delta': {'content': '各种问题。',
                        'function_call': None,
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916277,
 'id': '021792916276720c65202e590747a44f6b901a7b0655cff1a8bf6',
 'model': 'deepseek-v4-1-flash-261017',
 'object': 'chat.completion.chunk',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': 'stop',
              'index': 0,
              'logprobs': None}],
 'created': 1792916277,
 'id': '021792916276720c65202e590747a44f6b901a7b0655cff1a8bf6',
 'model': 'deepseek-v4-1-flash-261017',


#### 实时打印回复

In [27]:
completion = client.chat.completions.create(
    model="deepseek-v4-1-flash-261017", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    # 判断回复内容是否非空
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end='')  # 设置 end='' 实现不换行，视觉上拼接输出

我是DeepSeek，由深度求索公司创造的AI助手，可以帮你解答问题、处理文本、分析文件，随时为你提供

帮助！

#### 拼接回复

In [28]:
completion = client.chat.completions.create(
    model="deepseek-v4-1-flash-261017", # 3：模型标识（model_id）可能存在差异
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
content = ""
for chunk in completion:
    if chunk.choices[0].delta.content:
    	content += chunk.choices[0].delta.content
print(content)

我是DeepSeek，由深度求索公司创造的AI助手，致力于为你提供热情、细腻的帮助！


### 思考模式

In [29]:
# 单轮对话示例
completion = client.chat.completions.create(
    model="deepseek-v4-1-flash-261017", # 3：模型标识（model_id）可能存在差异
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 结构化打印每个数据块
for chunk in completion:
    pprint(chunk.model_dump())

{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'reasoning_content': '',
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916280,
 'id': '021792916280281efa62bfca375794f3a9b85f4d703cfec150899',
 'model': 'deepseek-v4-1-flash-261017',
 'object': 'chat.completion.chunk',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': '',
                        'function_call': None,
                        'reasoning_content': 'The',
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916280,
 'id': '02

{'choices': [{'delta': {'content': '问题、处理',
                        'function_call': None,
                        'reasoning_content': '',
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916280,
 'id': '021792916280281efa62bfca375794f3a9b85f4d703cfec150899',
 'model': 'deepseek-v4-1-flash-261017',
 'object': 'chat.completion.chunk',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': None}
{'choices': [{'delta': {'content': '文本',
                        'function_call': None,
                        'reasoning_content': '',
                        'refusal': None,
                        'role': 'assistant',
                        'tool_calls': None},
              'finish_reason': None,
              'index': 0,
              'logprobs': None}],
 'created': 1792916280,
 'id':

#### 实时打印回复

In [30]:
completion = client.chat.completions.create(
    model="deepseek-v4-1-flash-261017", # 3：模型标识（model_id）可能存在差异
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 实时打印模型回复的增量内容
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        print(reasoning, end='')
    if delta.content:
        print(delta.content, end='')


The user asks: "请用一句话介绍你自己" — "Please introduce yourself in one sentence." I should respond in Chinese with one sentence. Keep it simple and accurate.我是由深度求索公司创造的

DeepSeek AI助手，可以帮你解答问题、处理文本、分析文件，并乐于陪你一起探索各种想法。

#### 拼接回复

In [31]:
completion = client.chat.completions.create(
    model="deepseek-v4-1-flash-261017",
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': '请用一句话介绍你自己'}
    ],
    stream=True, # 开启流式输出
)

# 拼接回复
reasoning_content = ""
content = ""
for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    if delta is None:
        continue
    reasoning = getattr(delta, 'reasoning_content', None)

    # 两个字段独立处理，同一块可能同时含有推理和正文
    if reasoning:
        reasoning_content += reasoning
    if delta.content:
        content += delta.content
print(f"===== 模型推理过程 =====\n{reasoning_content}")
print(f"===== 模型回复 =====\n{content}")

===== 模型推理过程 =====
我们需要回答中文一句话介绍自己。简洁。用户要求一句话。需要避免说自己是AI? 可以说“我是一个乐于助人、能为你解答问题并提供建议的AI助手。” 一句话。保持中文。
===== 模型回复 =====
我是一个乐于助人、能为你解答问题并提供建议的AI助手。
